## Système de recommandation basé sur les profils utilisateurs

Ce notebook propose un système de recommandation d’images personnalisé pour chaque utilisateur, en se basant sur leurs préférences déclarées.

### Méthode utilisée : filtrage basé sur le contenu
Chaque image du dataset est comparée au profil utilisateur selon :
- Les couleurs dominantes de l’image
- Le format de l’image
- L’orientation de l’image
- Un bonus si l’image a déjà été aimée par l’utilisateur

Un **score de similarité** est calculé pour chaque image. Les 5 images ayant le score le plus élevé sont proposées à l’utilisateur.

Ce système constitue une première version d’un moteur de recommandation personnalisé basé sur des attributs visuels simples.

In [ ]:
import json
import pandas as pd
from urllib.parse import unquote
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

with open("../annotation/JSON/annotations.json", "r", encoding="utf-8") as f:
    annotations = json.load(f)

data = []
for img, details in annotations.items():
    try:
        img_name = unquote(img)
        width, height = details["metadata"]["size"]
        orientation = (
            "portrait" if height > width else
            "landscape" if width > height else
            "square"
        )
        colors = details.get("dominant_colors", [])
        if not colors:
            continue
        c1 = colors[0].lstrip("#")
        r, g, b = int(c1[0:2], 16), int(c1[2:4], 16), int(c1[4:6], 16)

        data.append({
            "image": img_name,
            "width": width,
            "height": height,
            "orientation": orientation,
            "r": r, "g": g, "b": b
        })
    except Exception as e:
        print(f"Ignoré {img}: {e}")

df = pd.DataFrame(data)
print("Images chargées :", len(df))

df["orientation_code"] = df["orientation"].map({
    "portrait": 0, "landscape": 1, "square": 2
})

features = df[["width", "height", "orientation_code", "r", "g", "b"]]
X = StandardScaler().fit_transform(features)

kmeans = KMeans(n_clusters=5, random_state=42)
df["cluster"] = kmeans.fit_predict(X)

df[["image", "cluster"]].to_csv("image_clusters_from_json.csv", index=False)
print("Clustering terminé. Résultats dans image_clusters_from_json.csv")


Images chargées : 79
✅ Clustering terminé. Résultats dans image_clusters_from_json.csv
